# Behavioral Patterns — Core

## What's covered

- What the behavioral family solves — distributing responsibilities between objects
- **Strategy** — make a family of algorithms interchangeable at runtime
- **Observer** — notify many objects when one object changes
- **Command** — wrap a request as an object, so it can be queued, logged, or undone
- **Template Method** — define the skeleton of an algorithm, leave specific steps to subclasses
- **Iterator** — walk a collection without exposing its internal structure
- Why these five carry most of the day-to-day weight in real code


## What the behavioral family solves

If creational patterns answer *who builds the object* and structural patterns answer *how the objects fit together*, the behavioral family answers **how the objects communicate and divide work**.

The eleven behavioral patterns are about three recurring problems:

- **Choosing what to do.** Different algorithms or behaviours fit different situations. Strategy and Template Method pick an algorithm; Command captures one as a value; State lets the algorithm change with the object's mode.
- **Talking to other objects.** Observer broadcasts changes; Mediator centralizes communication; Chain of Responsibility passes requests along until handled.
- **Walking over structure.** Iterator decouples traversal from collection shape; Visitor decouples operation from class hierarchy.

This notebook covers the five that show up most often: Strategy, Observer, Command, Template Method, and Iterator. Notebook 05 covers the more situational five — State, Chain of Responsibility, Mediator, Memento, Visitor — that earn their complexity less frequently but are sharp tools when the problem fits.


## Strategy

**Intent:** define a family of algorithms, encapsulate each one, and make them interchangeable. The algorithm varies independently from clients that use it.

**Where it shows up:** sort comparators, payment processors, discount rules, validation rules, route planners (shortest vs. fastest vs. avoid-tolls), compression codecs, retry policies. Anywhere "how do we do this" has more than one valid answer at runtime.

**The shape:** a strategy interface declares the algorithm's signature. Concrete strategies implement it. The context object holds a strategy reference and delegates to it. Swapping strategies is a single assignment.

**Python's take:** Strategy is the pattern that *most completely dissolves* into language features. A "strategy" in Python is usually just a function. The "interface" is a callable type alias. We already met this shape in notebook 01's open-closed example; here we name it and make the structural alternative explicit.

**Kotlin's take:** a `fun interface` (single-abstract-method interface) plus lambdas gives you the same compactness, with compile-time type safety.

**Strategy versus Template Method:** Strategy uses *composition* — the algorithm lives in a separate object you swap. Template Method uses *inheritance* — subclasses fill in steps of a fixed algorithm. Same goal, different mechanism.


### Python


In [ ]:
from typing import Callable
from dataclasses import dataclass


# A strategy is just a callable — no interface declaration needed.
RetryPolicy = Callable[[int], float]  # (attempt_number) -> seconds to wait


def no_retry(attempt: int) -> float:
    return 0.0


def fixed_delay(seconds: float) -> RetryPolicy:
    return lambda attempt: seconds


def exponential_backoff(base: float = 0.1, cap: float = 30.0) -> RetryPolicy:
    return lambda attempt: min(base * (2 ** attempt), cap)


@dataclass
class HttpClient:
    retry: RetryPolicy

    def call(self, url: str) -> str:
        # In real code this would loop on failure and sleep retry(attempt).
        # Here we just report what the policy would do.
        return f"GET {url} (retry waits: {[self.retry(i) for i in range(4)]})"


# Swap strategies by assignment — same client, different behaviour.
print(HttpClient(retry=no_retry).call("/api/health"))
print(HttpClient(retry=fixed_delay(2.0)).call("/api/health"))
print(HttpClient(retry=exponential_backoff()).call("/api/health"))


### Kotlin

```kotlin
fun interface RetryPolicy {
    fun delay(attempt: Int): Double
}

val noRetry = RetryPolicy { _ -> 0.0 }
fun fixedDelay(seconds: Double) = RetryPolicy { _ -> seconds }
fun exponentialBackoff(base: Double = 0.1, cap: Double = 30.0) =
    RetryPolicy { attempt -> minOf(base * (1 shl attempt), cap) }

class HttpClient(private val retry: RetryPolicy) {
    fun call(url: String): String {
        val waits = (0..3).map { retry.delay(it) }
        return "GET $url (retry waits: $waits)"
    }
}

println(HttpClient(noRetry).call("/api/health"))
println(HttpClient(fixedDelay(2.0)).call("/api/health"))
println(HttpClient(exponentialBackoff()).call("/api/health"))
```


**When not to use Strategy.** When the algorithm never changes. If there's only ever one implementation, the indirection is pure overhead. Reach for Strategy when the choice of algorithm is a real axis of variation — configurable per environment, per request, or per A/B test.


## Observer

**Intent:** define a one-to-many dependency between objects so that when one object changes state, all its dependents are notified and updated automatically.

**Where it shows up:** event systems, MVC view updates, pub/sub messaging, reactive state libraries (RxJava, Combine, MobX), spreadsheet recalculation, file-watching tools.

**The shape:** the *subject* (the thing being observed) maintains a list of *observers* and exposes `subscribe()` / `unsubscribe()`. When its state changes, it calls each observer's `update()` method with the relevant data.

**Two delivery models:**

- **Push** — the subject sends the new state with the notification (`update(new_value)`).
- **Pull** — the subject just signals "something changed" and observers query for what they need.

**Python's take:** keep a list of callbacks; iterate to notify. Or use a library — `blinker` for explicit signals, `asyncio.Event` for one-shot wake-ups, RxPY for full reactive streams.

**Kotlin's take:** `Delegates.observable` for property-level change notifications; `Flow` / `StateFlow` for the reactive-stream variant; a manual list of callbacks for simple cases.

**The hidden cost:** observers create implicit dependencies that aren't visible in the type signatures. A subject that broadcasts to ten unknown observers can be surprisingly hard to debug.


### Python


In [ ]:
from typing import Callable


PriceObserver = Callable[[str, float], None]


class StockTicker:
    def __init__(self):
        self._observers: list[PriceObserver] = []
        self._last_price: dict[str, float] = {}

    def subscribe(self, observer: PriceObserver) -> Callable[[], None]:
        self._observers.append(observer)
        # return an unsubscribe handle
        return lambda: self._observers.remove(observer)

    def update_price(self, symbol: str, price: float) -> None:
        old = self._last_price.get(symbol)
        if old == price:
            return
        self._last_price[symbol] = price
        for obs in list(self._observers):  # iterate a copy — observers may unsubscribe
            obs(symbol, price)


# Subscribers — each one decides what to do with the event.
def log_price(symbol: str, price: float) -> None:
    print(f"  [log] {symbol} = ${price:.2f}")


def alert_on_drop(symbol: str, price: float) -> None:
    if price < 100:
        print(f"  [ALERT] {symbol} dropped below $100 (now ${price:.2f})")


ticker = StockTicker()
unsubscribe_log = ticker.subscribe(log_price)
ticker.subscribe(alert_on_drop)

ticker.update_price("ACME", 120.50)
ticker.update_price("ACME", 95.25)   # triggers alert

unsubscribe_log()
ticker.update_price("ACME", 88.00)   # only alert fires now


### Kotlin

```kotlin
typealias PriceObserver = (String, Double) -> Unit

class StockTicker {
    private val observers = mutableListOf<PriceObserver>()
    private val lastPrice = mutableMapOf<String, Double>()

    fun subscribe(observer: PriceObserver): () -> Unit {
        observers.add(observer)
        return { observers.remove(observer) }
    }

    fun updatePrice(symbol: String, price: Double) {
        if (lastPrice[symbol] == price) return
        lastPrice[symbol] = price
        observers.toList().forEach { it(symbol, price) }   // copy: observers may unsubscribe
    }
}

val logPrice: PriceObserver = { sym, p -> println("  [log] $sym = $${"%.2f".format(p)}") }
val alertOnDrop: PriceObserver = { sym, p ->
    if (p < 100) println("  [ALERT] $sym dropped below \$100 (now $${"%.2f".format(p)})")
}

val ticker = StockTicker()
val unsub = ticker.subscribe(logPrice)
ticker.subscribe(alertOnDrop)

ticker.updatePrice("ACME", 120.50)
ticker.updatePrice("ACME", 95.25)

unsub()
ticker.updatePrice("ACME", 88.00)
```


**When not to use Observer.** When the dependencies are simple and explicit, just call the dependents directly. The pattern adds value when the *set* of dependents is unknown to the subject, or changes at runtime. The cost is implicit coupling: a subject that triggers fifteen observers across the codebase can be very hard to trace through. For complex event flows, consider a dedicated event bus or reactive library rather than rolling your own.


## Command

**Intent:** encapsulate a request as an object, allowing parametrization of clients, queuing of requests, logging, and support for undoable operations.

**Where it shows up:** undo/redo systems, task queues (Celery, Sidekiq), macro recorders, GUI button bindings, transaction logs, CQRS command handlers, scheduling.

**The shape:** a `Command` object captures everything needed to perform an operation — the receiver, the method, the arguments. The command has an `execute()` method, often with a paired `undo()`. Clients construct commands and either invoke them directly or hand them to an invoker (a queue, a button, a history stack).

**The win:** the request becomes a *value*. You can put commands in a list (history), serialize them (audit log), defer them (queue), reverse them (undo).

**Python's take:** any callable is already a command, in the simplest case. For commands with state (undo support, retries, metadata) a class or dataclass with `execute()` and `undo()` methods.

**Kotlin's take:** function references for simple cases; a `sealed interface Command` with concrete subtypes when you need pattern matching, serialization, or rich command metadata.


### Python


In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass


class TextEditor:
    def __init__(self):
        self.text = ""

    def insert(self, position: int, content: str) -> None:
        self.text = self.text[:position] + content + self.text[position:]

    def delete(self, position: int, length: int) -> str:
        removed = self.text[position:position + length]
        self.text = self.text[:position] + self.text[position + length:]
        return removed


class Command(ABC):
    @abstractmethod
    def execute(self) -> None: ...

    @abstractmethod
    def undo(self) -> None: ...


@dataclass
class InsertCommand(Command):
    editor: TextEditor
    position: int
    content: str

    def execute(self) -> None:
        self.editor.insert(self.position, self.content)

    def undo(self) -> None:
        self.editor.delete(self.position, len(self.content))


@dataclass
class DeleteCommand(Command):
    editor: TextEditor
    position: int
    length: int
    _removed: str = ""

    def execute(self) -> None:
        self._removed = self.editor.delete(self.position, self.length)

    def undo(self) -> None:
        self.editor.insert(self.position, self._removed)


# Invoker — keeps a history for undo.
class History:
    def __init__(self):
        self._stack: list[Command] = []

    def run(self, cmd: Command) -> None:
        cmd.execute()
        self._stack.append(cmd)

    def undo(self) -> None:
        if self._stack:
            self._stack.pop().undo()


editor = TextEditor()
history = History()

history.run(InsertCommand(editor, 0, "Hello, world!"))
print(repr(editor.text))      # 'Hello, world!'

history.run(DeleteCommand(editor, 5, 7))
print(repr(editor.text))      # 'Hello!'

history.undo()
print(repr(editor.text))      # 'Hello, world!'

history.undo()
print(repr(editor.text))      # ''


### Kotlin

```kotlin
class TextEditor {
    var text: String = ""
        private set

    fun insert(position: Int, content: String) {
        text = text.substring(0, position) + content + text.substring(position)
    }

    fun delete(position: Int, length: Int): String {
        val removed = text.substring(position, position + length)
        text = text.substring(0, position) + text.substring(position + length)
        return removed
    }
}

sealed interface Command {
    fun execute()
    fun undo()
}

class InsertCommand(
    private val editor: TextEditor,
    private val position: Int,
    private val content: String,
) : Command {
    override fun execute() = editor.insert(position, content)
    override fun undo()    { editor.delete(position, content.length) }
}

class DeleteCommand(
    private val editor: TextEditor,
    private val position: Int,
    private val length: Int,
) : Command {
    private var removed: String = ""
    override fun execute() { removed = editor.delete(position, length) }
    override fun undo()    = editor.insert(position, removed)
}

class History {
    private val stack = ArrayDeque<Command>()
    fun run(cmd: Command) { cmd.execute(); stack.addLast(cmd) }
    fun undo() { if (stack.isNotEmpty()) stack.removeLast().undo() }
}

val editor = TextEditor()
val history = History()
history.run(InsertCommand(editor, 0, "Hello, world!"))
println(editor.text)
history.run(DeleteCommand(editor, 5, 7))
println(editor.text)
history.undo()
println(editor.text)
history.undo()
println(editor.text)
```


**When not to use Command.** When the operation is a simple, immediate function call with no need for queuing, undoing, or logging. A direct method call is shorter and clearer than wrapping it in a command object. Command earns its keep when the request needs to *outlive* the moment it's issued — sitting in a queue, on a history stack, or in an audit log.


## Template Method

**Intent:** define the skeleton of an algorithm in a base class, deferring some steps to subclasses. Template Method lets subclasses redefine specific steps without changing the algorithm's structure.

**Where it shows up:** framework hooks (Django's `View.dispatch()` calling `get()` / `post()`), test fixture lifecycles (`setUp()` / `tearDown()`), ORM lifecycle callbacks (`before_save()` / `after_save()`), data pipelines (extract → transform → load), document generators (header → body → footer).

**The shape:** an abstract base class defines a concrete *template method* that calls a sequence of other methods. Some of those methods are abstract — subclasses must implement them. Some are *hooks* — concrete methods that do nothing by default and subclasses may override. The template method's order is fixed; only the steps vary.

**Strategy versus Template Method, revisited:**

- **Strategy** uses composition: you inject a different algorithm.
- **Template Method** uses inheritance: you subclass and override specific steps.

Strategy gives more flexibility (any combination of strategies, swappable at runtime). Template Method gives more structure (the framework controls the order). Pick Strategy when *the entire algorithm* varies; pick Template Method when *steps within a fixed algorithm* vary.

**Python's take:** abstract base class with `@abstractmethod` for required steps, plain methods for hooks. Or a function that accepts other functions as arguments — the same intent without the inheritance.

**Kotlin's take:** `abstract class` with abstract methods for required steps and `open` methods for hooks.


### Python


In [ ]:
from abc import ABC, abstractmethod


class DataPipeline(ABC):
    # The template method — fixed order, subclasses cannot change it.
    def run(self, source: str) -> str:
        raw = self.extract(source)
        clean = self.transform(raw)
        if self.should_validate():  # hook — subclasses may override
            self.validate(clean)
        return self.load(clean)

    @abstractmethod
    def extract(self, source: str) -> list[str]: ...

    @abstractmethod
    def transform(self, rows: list[str]) -> list[str]: ...

    @abstractmethod
    def load(self, rows: list[str]) -> str: ...

    # Hook — subclasses may override; default is to validate.
    def should_validate(self) -> bool:
        return True

    def validate(self, rows: list[str]) -> None:
        if not rows:
            raise ValueError("no rows after transform")


class CsvPipeline(DataPipeline):
    def extract(self, source: str) -> list[str]:
        return source.strip().split("\n")

    def transform(self, rows: list[str]) -> list[str]:
        return [r.upper() for r in rows]

    def load(self, rows: list[str]) -> str:
        return f"wrote {len(rows)} rows to database"


class FastPipeline(DataPipeline):
    def extract(self, source: str) -> list[str]:
        return source.split(",")

    def transform(self, rows: list[str]) -> list[str]:
        return [r.strip() for r in rows]

    def load(self, rows: list[str]) -> str:
        return f"wrote {len(rows)} rows fast"

    def should_validate(self) -> bool:
        return False  # skip validation for speed


print(CsvPipeline().run("alice\nbob\ncarol"))
print(FastPipeline().run("alice, bob, carol"))


### Kotlin

```kotlin
abstract class DataPipeline {
    // Template method — fixed order, subclasses cannot change it.
    fun run(source: String): String {
        val raw = extract(source)
        val clean = transform(raw)
        if (shouldValidate()) validate(clean)
        return load(clean)
    }

    protected abstract fun extract(source: String): List<String>
    protected abstract fun transform(rows: List<String>): List<String>
    protected abstract fun load(rows: List<String>): String

    // Hook — open means "may override but does not have to".
    protected open fun shouldValidate(): Boolean = true

    protected open fun validate(rows: List<String>) {
        require(rows.isNotEmpty()) { "no rows after transform" }
    }
}

class CsvPipeline : DataPipeline() {
    override fun extract(source: String) = source.trim().split("
")
    override fun transform(rows: List<String>) = rows.map { it.uppercase() }
    override fun load(rows: List<String>) = "wrote ${rows.size} rows to database"
}

class FastPipeline : DataPipeline() {
    override fun extract(source: String) = source.split(",")
    override fun transform(rows: List<String>) = rows.map { it.trim() }
    override fun load(rows: List<String>) = "wrote ${rows.size} rows fast"
    override fun shouldValidate() = false
}

println(CsvPipeline().run("alice
bob
carol"))
println(FastPipeline().run("alice, bob, carol"))
```


**When not to use Template Method.** When the algorithm's steps are simple enough that a function taking other functions as arguments would do the job. Template Method's overhead — abstract class, inheritance, protected methods — is justified when the framework needs to enforce the *order* and provide *multiple* extension points with sensible defaults. For a single point of variation, prefer Strategy with a passed-in function.


## Iterator

**Intent:** provide a way to access the elements of an aggregate object sequentially without exposing its underlying representation.

**Where it shows up:** *everywhere*. Iterator is so fundamental that modern languages have built it into the language itself — Python's `for x in xs:`, Java's `for (X x : xs)`, Kotlin's `for (x in xs)` all rely on the iterator protocol underneath.

**The shape:** an `Iterator` exposes two operations — "is there a next element?" and "give me the next element". The collection exposes a way to ask for an iterator over itself. The pattern decouples *traversal* from *collection shape* — code that walks a tree, a linked list, and a flat array looks the same.

**Python's take:** the iterator protocol — `__iter__()` returns an iterator, `__next__()` returns the next value or raises `StopIteration`. *Generators* (`yield`) are the idiomatic way to write custom iterators — they're functions that produce iterators with state preserved between calls.

**Kotlin's take:** the `Iterator<T>` interface with `hasNext()` / `next()`, but the *Sequence* abstraction (`sequence { yield(...) }`) gives you generator-style code with lazy evaluation.

**Why this pattern still matters even though it's built in:** when you write a non-trivial data structure — a tree, a sparse matrix, a custom graph — knowing the iterator protocol means you can plug your structure into every `for` loop, every list comprehension, every standard-library function that takes an iterable. The pattern *is* the interface.


### Python


In [ ]:
from dataclasses import dataclass, field
from typing import Iterator, Self


@dataclass
class TreeNode:
    value: int
    left: "TreeNode | None" = None
    right: "TreeNode | None" = None


# Approach 1 — generator function. The yield keyword keeps state across calls.
def in_order(node: TreeNode | None) -> Iterator[int]:
    if node is None:
        return
    yield from in_order(node.left)
    yield node.value
    yield from in_order(node.right)


# Approach 2 — implement the iterator protocol on a class.
class BinaryTree:
    def __init__(self, root: TreeNode):
        self.root = root

    def __iter__(self) -> Iterator[int]:
        return in_order(self.root)   # delegate to the generator


# Build a small binary-search tree.
#         5
#       /   \
#      3     8
#     / \    \
#    1   4    9
root = TreeNode(5,
    left=TreeNode(3, TreeNode(1), TreeNode(4)),
    right=TreeNode(8, right=TreeNode(9))
)

tree = BinaryTree(root)
print(list(tree))                    # [1, 3, 4, 5, 8, 9]  — in-order traversal
print([v for v in tree if v > 4])    # [5, 8, 9] — works in comprehensions
print(sum(tree))                     # 30 — works in standard-library functions


### Kotlin

```kotlin
data class TreeNode(
    val value: Int,
    val left: TreeNode? = null,
    val right: TreeNode? = null,
)

// Sequence builder — the Kotlin equivalent of a Python generator.
fun inOrder(node: TreeNode?): Sequence<Int> = sequence {
    if (node != null) {
        yieldAll(inOrder(node.left))
        yield(node.value)
        yieldAll(inOrder(node.right))
    }
}

class BinaryTree(private val root: TreeNode) : Iterable<Int> {
    override fun iterator(): Iterator<Int> = inOrder(root).iterator()
}

val root = TreeNode(5,
    left = TreeNode(3, TreeNode(1), TreeNode(4)),
    right = TreeNode(8, right = TreeNode(9))
)

val tree = BinaryTree(root)
println(tree.toList())                            // [1, 3, 4, 5, 8, 9]
println(tree.filter { it > 4 }.toList())          // [5, 8, 9]
println(tree.sum())                               // 30
```


**When not to use Iterator.** Almost never — the pattern is so fundamental it's the cost of admission for any data structure you want to share with the rest of the language ecosystem. The one case to flag is **eager versus lazy**: a generator produces values on demand and doesn't materialize a list. If your traversal is expensive and the consumer might not need every value, generators win. If you need to walk the same data many times, materializing once into a list is cheaper than re-running the generator.


## Picking the right behavioral-core pattern

| Question | Pattern |
|---|---|
| "I have several interchangeable algorithms — which one varies at runtime." | **Strategy** |
| "When one object changes, many others need to react." | **Observer** |
| "I need to queue, log, or undo operations." | **Command** |
| "I have a fixed algorithm with variable steps." | **Template Method** |
| "I need to walk a custom data structure without exposing its shape." | **Iterator** |

The two pairs worth distinguishing one more time:

- **Strategy versus Template Method.** Both vary the algorithm. Strategy varies *the whole algorithm* via composition (pass a different object). Template Method varies *steps within a fixed algorithm* via inheritance (override specific methods).
- **Observer versus Command.** Both decouple sender from receiver. Observer is *push-based* — the subject calls into observers when state changes. Command is *value-based* — the request becomes an object the receiver acts on when it's ready.

Strategy and Iterator are the two patterns that have most completely dissolved into modern languages. The vocabulary still matters when discussing intent; the structural ceremony is usually one line.


Notebook five turns to **Behavioral Patterns — Advanced**. The remaining five: State, Chain of Responsibility, Mediator, Memento, and Visitor. These show up less often, but each solves a specific problem cleanly when its shape matches yours.
